In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
import pickle

In [7]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [8]:
data = data.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [9]:
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [10]:
from sklearn.preprocessing import OneHotEncoder
# Initialize One-Hot Encoder
onehot_encoder_geo = OneHotEncoder(sparse_output=False)

# Fit and transform the Geography column
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])

# Convert to DataFrame
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography']),
    index=data.index
)

# Drop original Geography column and concatenate encoded columns
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [11]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [13]:
X = data.drop('EstimatedSalary', axis = 1)
y = data['EstimatedSalary']

In [14]:
X

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,1,0.0,1.0,0.0


In [15]:
y

0       101348.88
1       112542.58
2       113931.57
3        93826.63
4        79084.10
          ...    
9995     96270.64
9996    101699.77
9997     42085.58
9998     92888.52
9999     38190.78
Name: EstimatedSalary, Length: 10000, dtype: float64

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)   # Linear activation by default
])

In [19]:
model.compile(optimizer='adam', loss='mean_absolute_error',metrics=['mae'])

In [22]:
print(model.summary())

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [23]:
import datetime
from tensorflow.keras.callbacks import TensorBoard

log_dir = "logs/regression/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

In [24]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [25]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorboard_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 3s 6ms/step - loss: 100379.3281 - mae: 100379.3281 - val_loss: 98523.0625 - val_mae: 98523.0625
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 99639.7734 - mae: 99639.7734 - val_loss: 97009.0703 - val_mae: 97009.0703
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 96968.5391 - mae: 96968.5391 - val_loss: 93069.3828 - val_mae: 93069.3828
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 91658.3672 - mae: 91658.3672 - val_loss: 86467.3594 - val_mae: 86467.3594
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 83907.3750 - mae: 83907.3750 - val_loss: 77983.7734 - val_mae: 77983.7734
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 74809.4453 - mae: 74809.4453 - val_loss: 69233.9453 - val_mae: 69233.9453
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 65994.6484 - mae: 65994.648

In [26]:
%load_ext tensorboard

In [27]:
%tensorboard --logdir logs/regression

In [28]:
test_loss,test_mae = model.evaluate(X_test,y_test)
print(f'Test MAE: {test_mae}')

63/63 [==============================] - 0s 3ms/step - loss: 50335.1719 - mae: 50335.1719
Test MAE: 50335.171875
